<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_2_Confusion_Matrix_Basic_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Part 2
## The Confusion Matrix and Metrics

Author: Brad Sheese

---

## Introduction: Moving Beyond Accuracy

In the previous notebook, we saw that our Credit Default dataset is imbalanced (~70% good, ~30% bad). We learned that accuracy can be a trap, a model that does nothing but guess "Good" for everyone is 70% accurate, but it fails to catch any defaults. We expect our model to do better than just guessing the majority class everytime, so percentage of the majority class becomes the baseline accuracy that we compare our performance against.

In this notebook, we present the confusion matrix. Which will allow us to dissect a model's successes and failures using four metrics: accuracy, precision, recall, and f1-score.

### Learning Objectives
By the end of this notebook, you will be able to:
1. Build and interpret a confusion matrix.
2. Identify type I (false alarm) vs. type II (missed case) errors.
3. Calculate and interpret precision and recall in a business context.
4. Use the f1-score to balance the trade-off between different types of errors.

## Section 1: Setup and Baseline Model

We'll reload the German Credit Default data and retrain our XGBoost model using the `scale_pos_weight` parameter we introduced in Part 1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)

# Load and prepare data
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

y = (df['class'] == 'bad').astype(int) # converts class to 0, 1 using a boolean mask
X = df.drop(columns=['class']) # features are all cols in the dataframe minus the class

# train test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Update the XGBoost model parameters
model = xgb.XGBClassifier(
    n_estimators=100,           # Total number of trees (boosting rounds) to build
    max_depth=4,                # Maximum levels in each tree; limits model complexity/overfitting
    learning_rate=0.1,          # Step size shrinkage; prevents the model from over-adjusting to errors
    scale_pos_weight=scale_pos_weight, # Weights the minority class to handle imbalanced datasets
    eval_metric='logloss',      # The loss function used to measure model performance during training
    random_state=42,            # Ensures the same results are produced every time the code runs
    enable_categorical=True,    # Allows the model to process categorical features without manual encoding
    tree_method='hist'          # Uses histogram binning to speed up training on large datasets
)
# Trains the model by finding the patterns/weights that map features (X) to targets (y)
model.fit(X_train, y_train)

# Predicts the final discrete class labels (e.g., 0 or 1) using a 0.5 threshold
y_pred = model.predict(X_test)

# Predicts the raw probability scores (0.0 to 1.0) for the positive class (Column 1)
y_proba = model.predict_proba(X_test)[:, 1]

print("Setup complete. Model trained!")


Setup complete. Model trained!


## Section 2: The Confusion Matrix (The Error Map)

The confusion matrix is a table that maps the model's predictions against the actual outcomes. It allows us to see exactly where the "confusion" is happening.

### Terminology:
- **True Positives (TP):** We predicted Default, and they did default. *(Correct)*
- **True Negatives (TN):** We predicted Non-Default, and they did not default. *(Correct)*
- **False Positives (FP):** We predicted Default, but they did not actually default. *(False alarm / Type I error)*
- **False Negatives (FN):** We predicted Non-Default, but they actually defaulted. *(Missed case / Type II error)*

In [ ]:
# Generate the matrix, the matrix is a 2 X 2 array that matches the confusion matrix visualized below
cm = confusion_matrix(y_test, y_pred)

# Extract the results from the matrix
# .ravel() flattens the 2 x 2 array,
# then individual elements are assigned to variable names
tn, fp, fn, tp = cm.ravel()

print(f"--- Confusion Matrix Breakdown ---")
print(f"True Negatives  (TN) — Correctly Identified Non-Default: {tn}")
print(f"True Positives  (TP) — Correctly Identified Default:  {tp}")
print(f"False Positives (FP) — False Alarms / Type I:     {fp}")
print(f"False Negatives (FN) — Missed Defaults / Type II: {fn}")

# Visualize with labeled cells
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted: Non-Default', 'Predicted: Default'])
ax.set_yticklabels(['Actual: Non-Default', 'Actual: Default'])
ax.set_xlabel('Predicted Label')
ax.set_ylabel('Actual Label')
ax.set_title('Confusion Matrix: Credit Default Prediction')

# Add labels with both count and terminology
labels = [
    [f'TN: {tn}', f'FP: {fp}'],
    [f'FN: {fn}', f'TP: {tp}']
]
for i in range(2):
    for j in range(2):
        ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=14, fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### Interpreting the Results

The output above shows the exact count of each error type. The two error categories have different real-world consequences:

- **False positives (FP):** Non-defaulting customers who were wrongly flagged as risky. These are customers who would have repaid their loan, but the model denied them. The bank loses their interest income and potentially damages customer relationships.
- **False negatives (FN):** Actual defaulters the model missed and approved. These customers will not repay their loan. The bank loses the principal amount lent.

Which error costs the bank more? That depends on the problem. A conservative bank might prefer more false alarms (deny some good customers) over missed defaults (approve risky ones). A growth-oriented bank might accept more defaults to capture more good customers.

This asymmetry — where the two error types carry different costs — is exactly why choosing the right decision threshold matters. We will quantify these costs directly in Notebook 3.

## Section 3: Precision and Recall (The Trade-Off)

In Notebook 1 our model achieved accuracy above the 70% naive baseline, but not by much. Accuracy alone does not tell us which predictions are correct. To get a clearer picture of how the model handles the minority class, we use precision and recall.

### Precision: How reliable are our positive predictions?
When the model flags someone as a default risk, how often is it right?
$$\text{Precision} = \frac{TP}{TP + FP}$$
High precision means few false alarms.

### Recall: How many actual cases did we catch?
Of all the people who actually defaulted, what percentage did we catch?
$$\text{Recall} = \frac{TP}{TP + FN}$$
High recall means few missed cases.

---

### Two Ways to Remember Precision and Recall

#### 1. The Denominator Trick

The fastest way to reconstruct the formulas on an exam: look at what is in the denominator.

- **R**ecall has **R**eal positives in the denominator: $\dfrac{TP}{TP + FN}$ — all the cases that were *actually* positive.
- **P**recision has **P**redicted positives in the denominator: $\dfrac{TP}{TP + FP}$ — all the cases the model *predicted* as positive.

#### 2. The "Courtroom vs. Airport" Analogy

Think about which kind of error each system is designed to avoid:

- **The Courtroom (High Precision):** The legal system is built on the principle "innocent until proven guilty." We would rather let a guilty person go free (False Negative) than send an innocent person to jail (False Positive). The system is *picky* about who it convicts.
- **The Airport (High Recall):** TSA scanners are built to catch every threat. They do not care if they flag your belt buckle (False Positive), as long as they *never miss a weapon* (False Negative). The system casts a wide net to retrieve every possible threat.

The same trade-off applies to our credit model: being picky about flagging defaults means we will miss some real ones; casting a wide net means we will wrongly deny some good customers.

---

### F1-Score, Combining Precision and Recall

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

You might wonder: *Why use a complex formula instead of just taking the normal average of precision and recall?*

Imagine a model designed to predict loan defaults where **Precision = 100%** but **Recall = 1%**. This means the model flagged exactly one person as a defaulter and was absolutely right (perfect precision!), but it completely missed the other 99 actual defaulters.

If we simply averaged the two metrics:
* **The Arithmetic Mean** would be **50.5%** `(100 + 1) / 2`. This is incredibly misleading, as it makes a terrible model look "decent."
* **The F1-Score** would be roughly **2%**. This correctly exposes the model as virtually useless.

Because the F1-score calculates the **harmonic mean**, it strictly punishes extreme disparities between precision and recall. A model can only achieve a high F1-score if *both* precision and recall are strong.

In [ ]:
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {prec*100:.1f}% (Reliability of Default predictions)")
print(f"Recall:    {rec*100:.1f}% (Percentage of actual Defaults caught)")
print(f"F1-Score:  {f1*100:.1f}% (Balance between precision and recall)")

Precision: 54.6% (Reliability of Default predictions)
Recall:    58.9% (Percentage of actual Defaults caught)
F1-Score:  56.7% (Balance between precision and recall)


### What Do These Numbers Mean for Our Model?

- **Precision of ~55%:** When the model flags someone as a default risk, it is correct just over half the time. Nearly half of the people flagged will be good customers who get wrongly denied.
- **Recall of ~59%:** The model catches roughly 59% of actual defaulters — meaning it misses about 4 in 10 people who will eventually fail to repay.
- **F1-Score of ~57%:** The harmonic mean confirms that neither metric is particularly strong on its own.

These numbers reflect a genuinely difficult problem — human financial behavior is hard to predict from application data alone. The model is learning something real (it substantially beats random guessing), but both error rates leave significant room for improvement.

Whether these scores are "good enough" depends on the cost of each error type. A bank that loses $5,000 on a missed default but only $500 on a wrongly denied good customer should weight recall much more heavily than precision. That cost-based reasoning is exactly what Notebook 3 formalizes.




### The Full Classification Report





Scikit-learn provides `classification_report`, which shows precision, recall, and f1-score for each class along with the number of samples (support).

You might be wondering why the report prints two sets of precision, recall, and F1-scores.

When we first introduce these metrics, we frame them around one target: the "Positive" class (Default) and the "Negative" class (Non-Default). But `scikit-learn`'s classification report calculates the metrics from each class's perspective in turn — once treating Non-Default as the positive outcome, and once treating Default as the positive outcome. The same formulas apply both times; only the labeling changes.

### Breaking Down the Two Rows

**1. The "Non-Default" row (Class 0)**
Here, a correct "Non-Default" prediction is the positive hit.
- **Precision:** Of everyone the model predicted would repay, how many actually did?
- **Recall:** Of everyone who actually repaid, how many did the model correctly clear?
- **F1-Score:** The balance of our ability to identify good customers.

**2. The "Default" row (Class 1)**
Here, a correct "Default" prediction is the positive hit.
- **Precision:** Of everyone the model flagged as a risk, how many actually defaulted?
- **Recall:** Of everyone who actually defaulted, how many did the model catch?
- **F1-Score:** The balance of our ability to identify risky customers.

### Why is this useful?

The split report directly exposes class imbalance. Because 70% of applicants are good customers, the model finds it easy to score well on the Non-Default row. The Default row is where the real performance story lives — and where a weak model will reveal itself through poor recall.

### Other Metrics on the Report

- **Support:** The number of actual instances of each class in the test set.
- **Macro avg:** Treats both classes equally regardless of how many samples each has. Use this when you care about minority-class performance as much as majority-class performance.
- **Weighted avg:** Weights each class by its support, so it is closer to overall accuracy. Use this when sample counts should influence the summary.

For a credit default problem, the **Default (1) row** and the **macro avg** are the two most honest summaries of model performance.

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Non-Default (0)', 'Default (1)']))

## Section 4: Preparing for Notebook 4 — Threshold Tuning

Every metric in this notebook was computed at a single decision threshold: **0.5**. But 0.5 is an arbitrary default, not a principled choice for our problem.

### How the Threshold Shifts Precision and Recall

Moving the threshold changes which applicants get flagged:
- **Higher threshold (e.g., 0.7):** More conservative. The model only flags applicants it is very confident about. Precision rises (fewer false alarms), but recall falls (more defaulters are missed).
- **Lower threshold (e.g., 0.3):** More sensitive. The model flags more applicants. Recall rises (more defaulters caught), but precision falls (more good customers wrongly flagged).

This is not a flaw — it is a lever. The right setting depends on what errors cost your organization.

### `scale_pos_weight` vs. Threshold Tuning

In Notebook 1, we trained the model with `scale_pos_weight ≈ 2.33` to compensate for class imbalance. In Notebook 4, we take a different approach: train *without* `scale_pos_weight` and instead tune the decision threshold directly after training.

Both strategies address the same underlying problem — the model's tendency to ignore the minority class — but they do so at different stages:

| Strategy | Where it acts | Effect |
|---|---|---|
| `scale_pos_weight` | During training | Reweights the loss function so errors on the minority class cost more |
| Threshold tuning | At prediction time | Shifts the cutoff so more (or fewer) applicants are flagged as risky |

**You do not use both at the same time.** Notebook 4 uses threshold tuning because it gives more direct control and makes it straightforward to optimize for real business costs.

### What Notebook 4 Covers

Rather than evaluating at a single point, Notebook 4 will:
1. Plot the **ROC curve** — a picture of the TPR/FPR trade-off across *all* possible thresholds simultaneously.
2. Use the **AUC score** to compare models in a way that is independent of any single threshold.
3. Introduce the **precision-recall curve** as a more honest evaluation tool for imbalanced data.
4. Apply **Youden's J statistic** to find a mathematically optimal threshold.
5. Tune the threshold based on **real business costs**, where missing a default is far more expensive than a false alarm.

## Conclusion

We have introduced tools that provide a more honest evaluation of our model than accuracy alone. Key takeaways:

1. **The confusion matrix** breaks errors into four categories (TP, TN, FP, FN) and reveals exactly where the model fails — something accuracy cannot do.
2. **Precision** measures reliability: when the model says "default," how often is it right?
3. **Recall** measures coverage: of all actual defaulters, how many did we catch?
4. **F1-score** balances the two — it heavily penalizes models that are strong on one metric but weak on the other.
5. **All of these metrics depend on the 0.5 threshold**, which is arbitrary. The next notebook replaces single-point evaluation with tools that assess the model across every possible threshold simultaneously.

In Notebook 4, we will train a model *without* `scale_pos_weight`, plot the ROC and precision-recall curves, use Youden's J to find a principled threshold, and ultimately tune that threshold to minimize real business costs.